# 5-Class Emergency Department Triage Classifier: Optuna Hyperparameter Tuning & DeLong AUROC Confidence Intervals (`models/train_fedmm_classifier.ipynb`)

This notebook trains and tunes a **5-Class Multi-Class LightGBM Classifier** for Emergency Severity Index (ESI 1..5) triage on the **Federated Multi-Modal Emergency Dataset (`datasets/fedmml_ed_triage_dataset.csv`)** using 6 core vital signs and demographic features:

### 🔬 Methodology & Workflow
1. **Stratified 3-Way Partitioning (`config/triage_conf.json`)**:
   - Partitions encounters into Train, Validation, and Holdout Test sets dynamically based on `config/triage_conf.json` with target class stratification.
   - **Training Set**: Used for fitting MICE imputer, StandardScaler, and training candidate models.
   - **Validation Set**: Used exclusively by **Optuna** for hyperparameter evaluation and early stopping.
   - **Holdout Test Set**: Strictly held out for final benchmark evaluation.
2. **Leakage-Free MICE Imputation (`IterativeImputer`)**:
   - Imputes missing vital signs across `['systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2']` without discarding records. Fit exclusively on the Training set and applied to transform Validation and Test sets.
3. **Optuna Hyperparameter Tuning**:
   - Uses Tree-structured Parzen Estimators (`TPESampler`) to tune LightGBM parameters (`learning_rate`, `num_leaves`, `max_depth`, `min_child_samples`, `feature_fraction`, `bagging_fraction`, `bagging_freq`, `reg_alpha`, `reg_lambda`) targeting **Validation Macro Balanced Accuracy**.
4. **DeLong AUROC 95% Confidence Intervals**:
   - Calculates exact asymptotic standard errors and **95% Wald Confidence Intervals** for One-vs-Rest ROC-AUC curves on the held-out test set:
     $$\text{CI}_{95\%} = \Big[ \max\big(0, \, \widehat{\text{AUC}} - 1.96 \cdot \text{SE}_{\text{DeLong}}\big), \; \min\big(1, \, \widehat{\text{AUC}} + 1.96 \cdot \text{SE}_{\text{DeLong}}\big) \Big]$$

```mermaid
flowchart TD
    RawData["FedMML Dataset (87,234 encounters)"] --> Split["Stratified 3-Way Split via config/triage_conf.json (Train / Val / Test)"]
    Split --> MICE["MICE Imputation (fit on Train, transform Val & Test)"]
    MICE --> Scale["StandardScaler Normalization (fit on Train, transform Val & Test)"]
    
    Scale --> OptunaStudy["Optuna Study (TPESampler): Tune LightGBM on Validation Set<br/>Targeting Validation Macro Balanced Accuracy"]
    OptunaStudy --> BestParams["Extract Best Hyperparameters"]
    
    BestParams --> FinalTrain["Train Final Tuned LightGBM Model"]
    FinalTrain --> TestEval["Evaluate on Holdout Test Set: Per-Class Metrics & DeLong 95% AUROC CIs"]
    TestEval --> DiagPlots["Diagnostic Visualizations: 5x5 Confusion Matrix, ROC Curves with 95% CI, Feature Importance"]
```

In [ ]:
# ---------------------------------------------------------
# Step 1: Load FedMML Dataset, Encode 'sex', & Stratified 3-Way Split via triage_conf.json
# ---------------------------------------------------------
import os, json, pickle, time, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
import lightgbm as lgb
import optuna
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, recall_score,
    precision_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')
ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

# 1. Load Partitioning Configuration
config_path = f"{ROOT}/config/triage_conf.json"
with open(config_path, 'r') as f:
    config = json.load(f)

test_size = config['training']['test_size']
val_size  = config['training']['val_size']
seed_val  = config['training']['random_state']

print(f"Loaded Configuration from {config_path}:")
print(f"  * Test Size Fraction       = {test_size:.2f} ({test_size*100:.1f}%)")
print(f"  * Validation Size Fraction = {val_size:.2f} ({val_size*100:.1f}%)")
print(f"  * Random State Seed        = {seed_val}")
print("-" * 85)

# 2. Load Raw FedMML Dataset
data_path = f"{ROOT}/datasets/fedmml_ed_triage_dataset.csv"
print(f"Loading FedMML dataset from: {data_path}...")
df = pd.read_csv(data_path)
total_encounters = len(df)

required_features = ['age', 'sex', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2']
target_col = 'esi_level'

# 3. Encode 'sex' Feature (M -> 1, F -> 0)
df['sex_encoded'] = df['sex'].astype(str).str.strip().str.upper().map({'M': 1.0, 'MALE': 1.0, 'F': 0.0, 'FEMALE': 0.0})
feature_names = ['age', 'sex_encoded', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2']

df = df.dropna(subset=[target_col]).copy()
df[target_col] = df[target_col].astype(int)

print("=" * 85)
print(f"  FEDMML DATASET: {total_encounters:,} Total Encounters")
print("=" * 85)
print("Missing Value Count per Feature (to be imputed using MICE):")
print(df[feature_names].isnull().sum())
print("-" * 85)
print("Target ESI Distribution:")
esi_dist = df[target_col].value_counts().sort_index()
for cls_val, cnt in esi_dist.items():
    print(f"  * ESI Level {cls_val} : {cnt:,} encounters ({cnt/len(df)*100:.2f}%)")
print("=" * 85)

# 4. Stratified 3-Way Partitioning (Referencing triage_conf.json)
X_all = df[feature_names].values
y_all = df[target_col].values

# (a) Extract Stratified Holdout Test Set
X_rem_raw, X_test_raw, y_rem, y_test = train_test_split(
    X_all, y_all, test_size=test_size, stratify=y_all, random_state=seed_val
)

# (b) Extract Stratified Validation Set from remainder
val_adj_fraction = val_size / (1.0 - test_size)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_rem_raw, y_rem, test_size=val_adj_fraction, stratify=y_rem, random_state=seed_val + 1
)

print(f"Stratified Partition Complete (Referenced from triage_conf.json):")
print(f"  * Train Set      : {len(X_train_raw):,} encounters ({len(X_train_raw)/len(df)*100:.1f}%)")
print(f"  * Validation Set : {len(X_val_raw):,} encounters ({len(X_val_raw)/len(df)*100:.1f}%)")
print(f"  * Holdout Test   : {len(X_test_raw):,} encounters ({len(X_test_raw)/len(df)*100:.1f}%)")

In [ ]:
# ---------------------------------------------------------
# Step 2: MICE Imputation (IterativeImputer) & Feature Standardization
# ---------------------------------------------------------
print("=" * 85)
print("  MULTIVARIATE IMPUTATION BY CHAINED EQUATIONS (MICE)")
print("  (Fit exclusively on Training set to prevent validation/test leakage)")
print("=" * 85)

t0 = time.time()
mice_imputer = IterativeImputer(max_iter=10, random_state=42, verbose=0)

# 1. Fit MICE on training encounters and transform all sets
X_train_imp = mice_imputer.fit_transform(X_train_raw)
X_val_imp   = mice_imputer.transform(X_val_raw)
X_test_imp  = mice_imputer.transform(X_test_raw)

print(f"✓ MICE Imputation completed in {time.time()-t0:.2f}s!")
print(f"  - Train null count: {np.isnan(X_train_imp).sum()}")
print(f"  - Val null count  : {np.isnan(X_val_imp).sum()}")
print(f"  - Test null count : {np.isnan(X_test_imp).sum()}")
print("-" * 85)

# 2. Standardize Continuous Features (age, systolic_bp, heart_rate, respiratory_rate, spo2)
cont_indices = [0, 2, 3, 4, 5]

scaler = StandardScaler()
X_train = X_train_imp.copy()
X_val   = X_val_imp.copy()
X_test  = X_test_imp.copy()

X_train[:, cont_indices] = scaler.fit_transform(X_train_imp[:, cont_indices])
X_val[:, cont_indices]   = scaler.transform(X_val_imp[:, cont_indices])
X_test[:, cont_indices]  = scaler.transform(X_test_imp[:, cont_indices])

print(f"✓ Feature Normalization complete: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")

In [ ]:
# ---------------------------------------------------------
# Step 3: Vectorized DeLong Method for AUROC Variance & 95% Confidence Intervals
# Reference: DeLong et al. (1988), Biometrics 44(3):837-845
# ---------------------------------------------------------
def delong_roc_variance(ground_truth_binary, predictions_continuous, alpha=0.05):
    """
    Computes exact AUROC, DeLong asymptotic standard error (SE), and (1 - alpha)% confidence interval.
    Vectorized O(N log N) algorithm using sorted structural midranks.
    """
    pos = predictions_continuous[ground_truth_binary == 1]
    neg = predictions_continuous[ground_truth_binary == 0]
    m = len(pos)
    n = len(neg)
    
    if m == 0 or n == 0:
        return 0.0, 0.0, (0.0, 0.0), "[0.0000 - 0.0000]"
    
    pos_sorted = np.sort(pos)
    neg_sorted = np.sort(neg)
    
    # Structural components V10 and V01 via fast binary search
    v10 = (np.searchsorted(neg_sorted, pos, side='left') + np.searchsorted(neg_sorted, pos, side='right')) / (2.0 * n)
    v01 = 1.0 - (np.searchsorted(pos_sorted, neg, side='left') + np.searchsorted(pos_sorted, neg, side='right')) / (2.0 * m)
    
    auc = float(np.mean(v10))
    s10 = float(np.var(v10, ddof=1)) if m > 1 else 0.0
    s01 = float(np.var(v01, ddof=1)) if n > 1 else 0.0
    
    variance = (s10 / m) + (s01 / n)
    se = float(np.sqrt(max(0.0, variance)))
    
    z_crit = 1.959963984540054  # 95% Normal critical value
    ci_lower = max(0.0, auc - z_crit * se)
    ci_upper = min(1.0, auc + z_crit * se)
    ci_str = f"[{ci_lower:.4f} - {ci_upper:.4f}]"
    
    return auc, se, (ci_lower, ci_upper), ci_str

print("✓ Vectorized DeLong's Method for AUROC Variance & 95% CI loaded successfully!")

In [ ]:
# ---------------------------------------------------------
# Step 4: Optuna Hyperparameter Tuning on Validation Set
# ---------------------------------------------------------
print("=" * 85)
print("  OPTUNA HYPERPARAMETER TUNING (EVALUATING ON VALIDATION SET)")
print("=" * 85)

def objective_lgbm(trial):
    params = {
        'objective': 'multiclass',
        'num_class': 5,
        'metric': 'multi_logloss',
        'class_weight': 'balanced',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'n_estimators': 250,
        'verbosity': -1,
        'random_state': 42
    }
    
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_train, y_train - 1,
        eval_set=[(X_val, y_val - 1)],
        callbacks=[lgb.early_stopping(stopping_rounds=15, verbose=False)]
    )
    
    val_preds = model.predict(X_val) + 1
    val_bal_acc = balanced_accuracy_score(y_val, val_preds)
    return val_bal_acc

t_opt_start = time.time()
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective_lgbm, n_trials=30, timeout=180, show_progress_bar=False)

print(f"✓ Optuna Study Complete in {time.time()-t_opt_start:.1f}s across {len(study.trials)} trials!")
print(f"  * Best Trial Number           : #{study.best_trial.number}")
print(f"  * Best Validation Balanced Acc : {study.best_value*100:.2f}%")
print("\nOptimal Hyperparameters:")
for k, v in study.best_params.items():
    if isinstance(v, float):
        print(f"  - {k:20s}: {v:.6f}")
    else:
        print(f"  - {k:20s}: {v}")
print("=" * 85)

In [ ]:
# ---------------------------------------------------------
# Step 5: Train Final Tuned LightGBM Model
# ---------------------------------------------------------
best_lgb_params = {
    'objective': 'multiclass',
    'num_class': 5,
    'metric': 'multi_logloss',
    'class_weight': 'balanced',
    'n_estimators': 300,
    'verbosity': -1,
    'random_state': 42,
    **study.best_params
}

print("Training final tuned LightGBM model...")
t_final = time.time()
best_model = lgb.LGBMClassifier(**best_lgb_params)
best_model.fit(
    X_train, y_train - 1,
    eval_set=[(X_val, y_val - 1)],
    callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)]
)
print(f"✓ Final Tuned Model fitted in {time.time()-t_final:.1f}s (Best Iteration: {best_model.best_iteration_})!")

In [ ]:
# ---------------------------------------------------------
# Step 6: Holdout Test Set Evaluation with DeLong 95% Confidence Intervals
# ---------------------------------------------------------
probs_test = best_model.predict_proba(X_test)
preds_test = best_model.predict(X_test) + 1  # Map back {0..4} -> {1..5}

def compute_comprehensive_delong_metrics(y_true, y_pred, probs, pipeline_name):
    classes = [1, 2, 3, 4, 5]
    rows = []
    recalls, specs, bal_accs, precs, f1s, aucs, ses = [], [], [], [], [], [], []
    
    for idx, cls in enumerate(classes):
        y_bin_true = (y_true == cls).astype(int)
        y_bin_pred = (y_pred == cls).astype(int)
        
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal  = (rec + spec) / 2.0
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        f1   = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
        
        auc, se, ci, ci_str = delong_roc_variance(y_bin_true, probs[:, idx])
        
        recalls.append(rec); specs.append(spec); bal_accs.append(bal)
        precs.append(prec); f1s.append(f1); aucs.append(auc); ses.append(se)
        
        rows.append({
            'Pipeline': pipeline_name,
            'Class': f'ESI_{cls}',
            'Recall_Sensitivity': round(rec, 4),
            'Specificity': round(spec, 4),
            'Balanced_Accuracy': round(bal, 4),
            'Precision_PPV': round(prec, 4),
            'F1_Score': round(f1, 4),
            'AUROC': round(auc, 4),
            'DeLong_SE': round(se, 4),
            'DeLong_95_CI': ci_str
        })
    
    macro_auc = float(np.mean(aucs))
    macro_se  = float(np.sqrt(np.sum(np.array(ses)**2)) / len(classes))
    macro_ci_low = max(0.0, macro_auc - 1.96 * macro_se)
    macro_ci_up  = min(1.0, macro_auc + 1.96 * macro_se)
    macro_ci_str = f"[{macro_ci_low:.4f} - {macro_ci_up:.4f}]"
    
    rows.append({
        'Pipeline': pipeline_name,
        'Class': 'Macro_Average',
        'Recall_Sensitivity': round(np.mean(recalls), 4),
        'Specificity': round(np.mean(specs), 4),
        'Balanced_Accuracy': round(np.mean(bal_accs), 4),
        'Precision_PPV': round(np.mean(precs), 4),
        'F1_Score': round(np.mean(f1s), 4),
        'AUROC': round(macro_auc, 4),
        'DeLong_SE': round(macro_se, 4),
        'DeLong_95_CI': macro_ci_str
    })
    return pd.DataFrame(rows)

report_test_df = compute_comprehensive_delong_metrics(
    y_test, preds_test, probs_test, 'FedMML_Optuna_Tuned_LightGBM'
)

print("=" * 125)
print(f"   HOLDOUT TEST SET EVALUATION: OPTUNA-TUNED LIGHTGBM WITH DELONG AUROC 95% CI (N = {len(y_test):,})")
print("=" * 125)
print(report_test_df[['Class', 'Recall_Sensitivity', 'Specificity', 'Balanced_Accuracy', 'Precision_PPV', 'F1_Score', 'AUROC', 'DeLong_SE', 'DeLong_95_CI']].to_string(index=False))
print("=" * 125 + chr(10))

# Export CSV Report
reports_dir = f"{ROOT}/reports"
os.makedirs(reports_dir, exist_ok=True)
report_path = os.path.join(reports_dir, 'fedmml_lightgbm_optuna_delong_report.csv')
report_test_df.to_csv(report_path, index=False)
print(f"✓ Performance report saved to: {report_path}")

In [ ]:
# ---------------------------------------------------------
# Step 7: Diagnostic Visualizations (Confusion Matrix, ROC Curves with DeLong CIs & Feature Importance)
# ---------------------------------------------------------
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

plots_dir = f"{ROOT}/plots"
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

esi_labels = [f"ESI {i}" for i in range(1, 6)]

# 1. Test Set 5x5 Normalized Confusion Matrix
cm      = confusion_matrix(y_test, preds_test, labels=[1, 2, 3, 4, 5])
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

annot = np.empty_like(cm, dtype=object)
for i in range(5):
    for j in range(5):
        annot[i, j] = f"{cm[i, j]:,}\n({cm_norm[i, j]*100:.1f}%)"

fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(
    cm_norm, annot=annot, fmt='', cmap='Blues', cbar=True, ax=ax,
    vmin=0, vmax=1, xticklabels=esi_labels, yticklabels=esi_labels
)
macro_bal = report_test_df.loc[5, 'Balanced_Accuracy']
macro_auc_val = report_test_df.loc[5, 'AUROC']
macro_ci_text = report_test_df.loc[5, 'DeLong_95_CI']

ax.set_title(
    f"Holdout Test Set Confusion Matrix (Optuna-Tuned LightGBM)\n"
    f"Macro Balanced Acc: {macro_bal*100:.2f}% | Macro AUC: {macro_auc_val:.4f} {macro_ci_text}",
    fontsize=11.5, fontweight='bold', pad=12
)
ax.set_xlabel("Predicted ESI Level", fontsize=11, fontweight='bold')
ax.set_ylabel("True ESI Level", fontsize=11, fontweight='bold')

plt.tight_layout()
cm_path = os.path.join(plots_dir, "fedmml_lightgbm_tuned_confusion_matrix.png")
plt.savefig(cm_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'fedmml_lightgbm_tuned_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Confusion Matrix saved to: {cm_path}")

# 2. Multiclass One-vs-Rest ROC-AUC Curves with DeLong 95% Confidence Intervals
classes = [1, 2, 3, 4, 5]
n_classes = len(classes)
y_test_bin = label_binarize(y_test, classes=classes)
esi_colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']

fpr, tpr, roc_aucs = dict(), dict(), dict()
for i, cls in enumerate(classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], probs_test[:, i])
    roc_aucs[i] = auc(fpr[i], tpr[i])

fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), probs_test.ravel())
roc_aucs["micro"] = auc(fpr["micro"], tpr["micro"])

all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= n_classes
fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr
roc_aucs["macro"] = auc(fpr["macro"], tpr["macro"])

plt.figure(figsize=(9.5, 8))
plt.plot(fpr["micro"], tpr["micro"], label=f"Micro-Average (AUC = {roc_aucs['micro']:.4f})", color='#e377c2', linestyle=':', linewidth=2.5)
plt.plot(fpr["macro"], tpr["macro"], label=f"Macro-Average (AUC = {macro_auc_val:.4f}, 95% CI {macro_ci_text})", color='#17becf', linestyle='--', linewidth=2.5)

for i in range(5):
    cls_ci = report_test_df.loc[i, 'DeLong_95_CI']
    plt.plot(fpr[i], tpr[i], color=esi_colors[i], linewidth=2.0, label=f"ESI {i+1} (AUC = {roc_aucs[i]:.4f}, 95% CI {cls_ci})")

plt.plot([0, 1], [0, 1], 'k--', color='gray', linewidth=1.2, label='Random Guess (AUC = 0.5000)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=11, fontweight='bold')
plt.ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=11, fontweight='bold')
plt.title(f"Holdout Test Set Multiclass ROC-AUC Curves\n(Optuna-Tuned LightGBM with DeLong 95% CIs)", fontsize=12, fontweight='bold', pad=10)
plt.legend(loc="lower right", fontsize=9.2, frameon=True, framealpha=0.95)
plt.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
roc_path = os.path.join(plots_dir, "fedmml_lightgbm_tuned_roc_auc_curve.png")
plt.savefig(roc_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'fedmml_lightgbm_tuned_roc_auc_curve.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ ROC-AUC Curves with DeLong CIs saved to: {roc_path}")

# 3. Feature Importance Visualization
fig, ax = plt.subplots(figsize=(9, 5))
feat_imp = pd.DataFrame({
    'Feature': feature_names,
    'Importance_Gain': best_model.booster_.feature_importance(importance_type='gain'),
    'Importance_Split': best_model.booster_.feature_importance(importance_type='split')
}).sort_values('Importance_Gain', ascending=False)

sns.barplot(data=feat_imp, x='Importance_Gain', y='Feature', palette='Blues_r', ax=ax, edgecolor='black')
ax.set_title('Tuned LightGBM Feature Importance (Information Gain on FedMML Features)', fontsize=12, fontweight='bold', pad=10)
ax.set_xlabel('Total Information Gain', fontsize=11, fontweight='bold')
ax.set_ylabel('Feature Name', fontsize=11, fontweight='bold')

for p in ax.patches:
    ax.annotate(f"{p.get_width():,.1f}", (p.get_width(), p.get_y() + p.get_height() / 2.),
                ha='left', va='center', fontsize=9.5, fontweight='bold', xytext=(5, 0), textcoords='offset points')

plt.tight_layout()
feat_imp_path = os.path.join(plots_dir, "fedmml_lightgbm_tuned_feature_importance.png")
plt.savefig(feat_imp_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'fedmml_lightgbm_tuned_feature_importance.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Feature Importance plot saved to: {feat_imp_path}")

In [ ]:
# ---------------------------------------------------------
# Step 8: Export Production Artifact Bundle & Metadata Manifest
# ---------------------------------------------------------
deploy_dir = f"{ROOT}/deploy"
os.makedirs(deploy_dir, exist_ok=True)

bundle = {
    'mice_imputer': mice_imputer,
    'scaler': scaler,
    'feature_names': feature_names,
    'model': best_model,
    'best_params': study.best_params,
    'best_val_score': study.best_value,
    'config_training': config['training']
}

bundle_file = os.path.join(deploy_dir, 'fedmml_lightgbm_bundle.pkl')
with open(bundle_file, 'wb') as f:
    pickle.dump(bundle, f)

manifest = dict(
    pipeline='FedMML_Optuna_Tuned_LightGBM_Classifier',
    dataset='datasets/fedmml_ed_triage_dataset.csv',
    config_file='config/triage_conf.json',
    config_training=config['training'],
    imputation_method='MICE_IterativeImputer (fit on Train partition)',
    tuning_framework='Optuna (TPESampler)',
    tuning_objective='Validation Macro Balanced Accuracy',
    best_hyperparameters=study.best_params,
    best_validation_balanced_accuracy=study.best_value,
    auroc_confidence_intervals='DeLong non-parametric U-statistic method (95% Wald CI)',
    total_encounters=len(df),
    train_encounters=len(X_train),
    val_encounters=len(X_val),
    test_encounters=len(X_test),
    features=['age', 'sex', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2'],
    target='esi_level',
    n_classes=5,
    classes=['ESI 1', 'ESI 2', 'ESI 3', 'ESI 4', 'ESI 5'],
    holdout_test_metrics=report_test_df.to_dict(orient='records')
)

manifest_file = os.path.join(deploy_dir, 'fedmml_lightgbm_manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Production Bundle   : {bundle_file}")
print(f"✓ Production Manifest : {manifest_file}")